## Implementing Householder's method for QR decomposition

In [1]:
import numpy as np

def qrfactor(A):
    """
    Computes the QR factorization of a matrix A using
    Householder reflections.

    Based on the MATLAB code provided.
    """

    # --- 1. Initialization ---

    # In Python, arrays are passed by reference, so we make a copy
    # to avoid changing the original matrix 'A' outside the function.
    # We also ensure it's a float type for calculations.
    A = A.copy().astype(float)

    # Get shape. m = rows, n = columns
    # MATLAB: [m,n] = size(A);
    m, n = A.shape

    # Create an m x m identity matrix
    # MATLAB: Q = eye(m);
    Q = np.eye(m)

    # --- 2. Main Loop ---

    # Loop over columns.
    # MATLAB 'for k = 1:n' is 1-indexed and inclusive (1, 2, ..., n)
    # Python 'range(n)' is 0-indexed and exclusive (0, 1, ..., n-1)
    for k in range(n):

        # --- 3. Find the Householder Reflector ---

        # Get the k-th column from row k to the end
        # MATLAB: z = A(k:m, k);
        # Python: A[k:m, k] (or A[k:, k])
        z = A[k:m, k]

        # Calculate the Householder vector 'v'
        # This is a direct translation of the MATLAB logic:
        # v = [-sign(z(1))*norm(z) - z(1); -z(2:end)];

        # Start with v = -z.
        # We use .copy() so 'v' is a new vector, not just a view of 'z'
        v = -z.copy()
        norm_z = np.linalg.norm(z)

        # Handle the sign logic. np.sign(0) is 0, so this is safe.
        v[0] = -np.sign(z[0]) * norm_z - z[0]

        # --- 4. Normalize the Reflector Vector 'v' ---

        # Get the squared norm (v' * v)
        # MATLAB: v'*v
        v_dot_v = v @ v  # '@' is the dot product operator

        # Check for a zero vector to avoid division by zero
        if v_dot_v < 1e-16: # Use a small epsilon for float precision
            continue # This column is already zeroed, skip this iteration

        # Normalize v
        # MATLAB: v = v / sqrt(v'*v);
        v = v / np.sqrt(v_dot_v)

        # --- 5. Apply the Reflection (Vectorized) ---

        # The original code has 'for j' loops to apply the reflection
        # to each column of A and Q. In NumPy, we can do this all
        # at once with vectorized matrix operations.

        # Apply reflection to the remaining submatrix of A
        # (columns k through n)
        # This replaces the loop 'for j = 1:n'
        sub_A = A[k:m, k:n]
        dot_prods_A = v @ sub_A  # (v' * sub_A)
        # np.outer(v, dot_prods_A) creates the rank-1 update matrix
        A[k:m, k:n] = sub_A - 2 * np.outer(v, dot_prods_A)

        # Apply reflection to all columns of Q
        # This replaces the loop 'for j = 1:m'
        sub_Q = Q[k:m, :]
        dot_prods_Q = v @ sub_Q  # (v' * sub_Q)
        Q[k:m, :] = sub_Q - 2 * np.outer(v, dot_prods_Q)

    # --- 6. Finalize Q and R ---

    # Q was accumulating (Q_n * ... * Q_1 * I) which is Q_actual.T
    # MATLAB: Q = Q';
    Q = Q.T

    # R is the upper triangular part of the modified A
    # MATLAB: R = triu(A);
    R = np.triu(A)

    # Python functions return with the 'return' keyword
    return Q, R

# --- Example Usage ---
if __name__ == '__main__':
    # Create a sample matrix
    A = np.array([
        [1, 19, -34],
        [-2, -5, 20],
        [2, 8, 37],
        [1, 1, 1]
    ], dtype=float)

    Q, R = qrfactor(A)

    print("--- A ---")
    print(A)

    print("\n--- Q (Orthogonal) ---")
    print(Q)

    print("\n--- R (Upper Triangular) ---")
    print(np.round(R, 6)) # Round for cleaner display

    print("\n--- Q @ R (Should equal A) ---")
    print(np.round(Q @ R, 6)) # Round to account for float errors

    print("\n--- Q.T @ Q (Should be Identity) ---")
    print(np.round(Q.T @ Q, 6))

--- A ---
[[  1.  19. -34.]
 [ -2.  -5.  20.]
 [  2.   8.  37.]
 [  1.   1.   1.]]

--- Q (Orthogonal) ---
[[-0.31622777 -0.93068008  0.14993533  0.10655507]
 [ 0.63245553 -0.27144836 -0.61942397  0.37766352]
 [-0.63245553  0.07755667 -0.75951799 -0.13083343]
 [-0.31622777  0.23267002  0.1302527   0.91043885]]

--- R (Upper Triangular) ---
[[ -3.162278 -14.546477  -0.316228]
 [  0.       -15.472556  29.316423]
 [  0.         0.       -45.458194]
 [  0.         0.         0.      ]]

--- Q @ R (Should equal A) ---
[[  1.  19. -34.]
 [ -2.  -5.  20.]
 [  2.   8.  37.]
 [  1.   1.   1.]]

--- Q.T @ Q (Should be Identity) ---
[[ 1. -0.  0.  0.]
 [-0.  1.  0. -0.]
 [ 0.  0.  1. -0.]
 [ 0. -0. -0.  1.]]


## Code without comment

In [3]:
import numpy as np

def qrfactor(A):
    A = A.copy().astype(float)
    m, n = A.shape
    Q = np.eye(m)

    for k in range(n):

        z = A[k:m, k]

        v = -z.copy()
        norm_z = np.linalg.norm(z)
        v[0] = -np.sign(z[0]) * norm_z - z[0]

        v_dot_v = v @ v

        if v_dot_v < 1e-16:
            continue

        v = v / np.sqrt(v_dot_v)

        sub_A = A[k:m, k:n]
        dot_prods_A = v @ sub_A
        A[k:m, k:n] = sub_A - 2 * np.outer(v, dot_prods_A)

        sub_Q = Q[k:m, :]
        dot_prods_Q = v @ sub_Q
        Q[k:m, :] = sub_Q - 2 * np.outer(v, dot_prods_Q)

    Q = Q.T
    R = np.triu(A)

    return Q, R

In [9]:
if __name__ == '__main__':
    # Create a sample matrix
    A = np.array([
        [1, 19, -34],
        [-2, -5, 20],
        [2, 8, 37],
        [1, 1, 1]
    ], dtype=float)

    Q, R = qrfactor(A)

    print("--- A ---")
    print(A)

    print("\n--- Q (Orthogonal) ---")
    print(Q)

    print("\n--- R (Upper Triangular) ---")
    print(np.round(R, 6)) # Round for cleaner display

    print("\n--- Q @ R (Should equal A) ---")
    print(np.round(Q @ R, 6)) # Round to account for float errors

    print("\n--- Q.T @ Q (Should be Identity) ---")
    print(np.round(Q.T @ Q, 6))



--- A ---
[[  1.  19. -34.]
 [ -2.  -5.  20.]
 [  2.   8.  37.]
 [  1.   1.   1.]]

--- Q (Orthogonal) ---
[[-0.31622777 -0.93068008  0.14993533  0.10655507]
 [ 0.63245553 -0.27144836 -0.61942397  0.37766352]
 [-0.63245553  0.07755667 -0.75951799 -0.13083343]
 [-0.31622777  0.23267002  0.1302527   0.91043885]]

--- R (Upper Triangular) ---
[[ -3.162278 -14.546477  -0.316228]
 [  0.       -15.472556  29.316423]
 [  0.         0.       -45.458194]
 [  0.         0.         0.      ]]

--- Q @ R (Should equal A) ---
[[  1.  19. -34.]
 [ -2.  -5.  20.]
 [  2.   8.  37.]
 [  1.   1.   1.]]

--- Q.T @ Q (Should be Identity) ---
[[ 1. -0.  0.  0.]
 [-0.  1.  0. -0.]
 [ 0.  0.  1. -0.]
 [ 0. -0. -0.  1.]]


In [12]:
result = np.linalg.qr(A)
result.Q

array([[-0.31622777, -0.93068008,  0.14993533],
       [ 0.63245553, -0.27144836, -0.61942397],
       [-0.63245553,  0.07755667, -0.75951799],
       [-0.31622777,  0.23267002,  0.1302527 ]])

In [17]:
B = np.array([1,2,3,4,5,6,7,8]).reshape(4,2)
print(B)

[[1 2]
 [3 4]
 [5 6]
 [7 8]]


In [15]:
print(np.linalg.qr(B).Q)

[[-0.19611614 -0.98058068]
 [-0.98058068  0.19611614]]


In [18]:
qrfactor(B)[0]

array([[-0.10910895, -0.82951506, -0.39450102, -0.37995913],
       [-0.32732684, -0.43915503,  0.24279655,  0.80065588],
       [-0.54554473, -0.048795  ,  0.69790998, -0.46143436],
       [-0.76376262,  0.34156503, -0.5462055 ,  0.04073761]])

In [1]:
A = [[1,-1,4],[1,4,-2],[1,4,2],[1,-1,0]]

In [5]:
print(np.linalg.qr(A))
Q,R = np.linalg.qr(A)

QRResult(Q=array([[-0.5,  0.5, -0.5],
       [-0.5, -0.5,  0.5],
       [-0.5, -0.5, -0.5],
       [-0.5,  0.5,  0.5]]), R=array([[-2., -3., -2.],
       [ 0., -5.,  2.],
       [ 0.,  0., -4.]]))


In [9]:
Q@R

array([[ 1.0000000e+00, -1.0000000e+00,  4.0000000e+00],
       [ 1.0000000e+00,  4.0000000e+00, -2.0000000e+00],
       [ 1.0000000e+00,  4.0000000e+00,  2.0000000e+00],
       [ 1.0000000e+00, -1.0000000e+00, -4.4408921e-16]])

In [ ]:
Q